In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [0]:
schema = StructType([
    StructField("ride_id", StringType()),
    StructField("rideable_type", StringType()),
    StructField("started_at", TimestampType()),
    StructField("ended_at", TimestampType()),
    StructField("start_station_name", StringType()),
    StructField("start_station_id", StringType()),
    StructField("end_station_name", StringType()),
    StructField("end_station_id", StringType()),
    StructField("start_lat", DoubleType()),
    StructField("start_lng", DoubleType()),
    StructField("end_lat", DoubleType()),
    StructField("end_lng", DoubleType()),
    StructField("member_casual", StringType())
])

In [0]:
df = spark.read.format('csv')\
    .option("header", True)\
    .option("schema", schema)\
    .option("path", f"/Volumes/{catalog}/00_landing/source_citybike_data/JC-202503-citibike-tripdata.csv")\
    .load()

In [0]:
df = df.withColumn("metadata",
                   create_map(
                       lit("pipeline_id"), lit(pipeline_id),
                       lit("run_id"), lit(run_id),
                       lit("task_id"), lit(task_id),
                       lit("processed_timestamp"), lit(processed_timestamp)
                   ))

In [0]:
df.write\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.01_bronze.jc_citybike")